# Titanic ② 軽めの機械学習モデル

01 の性別ルール（train での正解率 0.787）を，機械学習で超えるのが目標．

特徴量は元の列をほぼそのまま使い，前処理も最小限にとどめる．名前や客室番号から新しい特徴量を作る工夫は，Discussion や他の人のノートブックを読んでからにする．

## データの準備

01 と同じく，`src/titanic/data.py` の関数で読み込む．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from titanic.data import load_data, save_submission

# print で DataFrame を出すときに，列が「...」で省略されないようにする
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

train, test = load_data()
print("train:", train.shape, " test:", test.shape)

## 特徴量と前処理

| 種類 | 列 | 前処理 |
|---|---|---|
| 数値 | Age, SibSp, Parch, Fare | 欠損を中央値で埋めて，標準化する |
| カテゴリ | Pclass, Sex, Embarked | 欠損を最頻値で埋めて，one-hot にする |
| 使わない | PassengerId, Name, Ticket, Cabin | ID や文字列そのままでは使いにくい列．Cabin は 8 割近くが欠損 |

- **中央値で埋める**: Age は train で 177 件，Fare は test で 1 件欠けている．平均より外れ値に引っ張られにくい中央値を使う．
- **Pclass を one-hot にする**: 1 < 2 < 3 という順序はあるが，1 等と 2 等の差と，2 等と 3 等の差が同じとは限らない．数値のまま入れると，ロジスティック回帰は「等級が 1 つ違うごとに同じだけ効く」と仮定してしまう．
- **標準化する**: ロジスティック回帰の正則化（係数が大きくなりすぎないようにかける罰則）は，特徴量のスケールがそろっていないと一部の特徴量に偏ってかかる．ランダムフォレストは値の大小関係しか見ないので，標準化しても結果は変わらない．そこで両方のモデルに同じ前処理を使う．

### リークを防ぐ

中央値や，標準化に使う平均・分散は，学習に使う行だけから計算しないといけない．train 全体で計算してから交差検証すると，検証用の行の情報が学習側に漏れて（リーク），スコアが実力より良く出てしまう．

前処理とモデルを Pipeline にまとめておけば，交差検証の分割ごとに，学習用の行だけを使って前処理の値を計算してくれる．

In [ ]:
NUMERIC = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL = ["Pclass", "Sex", "Embarked"]

X = train[NUMERIC + CATEGORICAL]
y = train["Survived"]
X_test = test[NUMERIC + CATEGORICAL]


def build_pipeline(model) -> Pipeline:
    """前処理とモデルをつないだ Pipeline を作る．"""
    preprocess = ColumnTransformer(
        [
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
            ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CATEGORICAL),
        ],
        # 変換後の列名を "num__Age" ではなく "Age" のようにする
        verbose_feature_names_out=False,
    )
    return make_pipeline(preprocess, model)

## 交差検証でモデルを比べる

train を 5 つに分け，4 つで学習して残りの 1 つで正解率を測る．検証に使う部分を入れ替えながらこれを 5 回繰り返し，平均をとる（5 分割交差検証）．`StratifiedKFold` を使うと，どの分割でも生存者の割合が train 全体とほぼ同じになる．

LB に出してスコアを見るのではなく手元で検証するのは，Titanic の LB がテストデータ全体で計算されるからでもある．LB を見ながら調整を重ねると，テストデータに合わせ込んだだけのモデルになりやすい．提出は 1 日 10 回までという制限もある．

比べるモデルは 2 つ．比較の基準として，01 の性別ルールも同じ分割で評価する．

- **ロジスティック回帰**: 特徴量の重み付き和をシグモイド関数で 0〜1 の確率に変える線形モデル．どの特徴量がどちら向きに効いたかを係数で読める．
- **ランダムフォレスト**: 少しずつ違うデータで決定木をたくさん作り，多数決で予測するモデル．「3 等の女性」のような特徴量の組み合わせ（交互作用）を自動で拾える．木を深くしすぎると train を丸暗記してしまうので，深さは 6 までにしておく．

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
models = {
    "ロジスティック回帰": LogisticRegression(max_iter=1000),
    "ランダムフォレスト": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0),
}

scores = {
    # 性別ルールは学習しないので，検証用の行で正解率を測るだけ
    "性別ルール": np.array([
        ((X.iloc[valid]["Sex"] == "female").astype(int) == y.iloc[valid]).mean()
        for _, valid in cv.split(X, y)
    ]),
}
for name, model in models.items():
    scores[name] = cross_val_score(build_pipeline(model), X, y, cv=cv, scoring="accuracy")

summary = pd.DataFrame({name: [s.mean(), s.std()] for name, s in scores.items()}, index=["平均", "標準偏差"]).T
print(summary.round(3))

ランダムフォレストが 0.825 で，性別ルール（0.787）をはっきり上回った．ロジスティック回帰は 0.796 で，性別ルールとの差は 0.009 しかない．分割ごとのばらつき（標準偏差 0.03〜0.05）より小さいので，これだけでは良くなったとは言えない．

差がついたのは，性別と等級の組み合わせを拾えるかどうかだと考えている．01 で見たとおり，1 等から 3 等になると女性の生存率は 0.97 から 0.50 まで下がるのに，男性は 0.37 から 0.14 への低下にとどまる．等級の効き方が男女でかなり違う．ロジスティック回帰は特徴量ごとの効果を足し合わせるだけのモデルなので，等級の効果は男女共通の 1 つの値しか持てない．決定木なら「性別で分けてから等級で分ける」という枝分かれを作れるので，この違いを表せる．

## ロジスティック回帰の係数を見る

係数がプラスなら生存，マイナスなら死亡の方向に効く．数値の特徴量は標準化してあるので，係数の大きさもある程度比べられる．

In [ ]:
logreg = build_pipeline(models["ロジスティック回帰"]).fit(X, y)
preprocess, model = logreg[0], logreg[-1]
coef = pd.Series(model.coef_[0], index=preprocess.get_feature_names_out()).sort_values()
print(coef.round(3))

いちばん効いているのは性別で，次が等級（1 等がプラス，3 等がマイナス）．年齢はマイナスなので，若いほど生存寄りになる．01 で見た傾向と合っている．

Sex_female と Sex_male は，片方が 1 ならもう片方は 0 になる列なので，係数はほぼ同じ大きさで符号が逆になる．Pclass_1 と Pclass_3 の関係も似ている．

意外だったのは Fare の係数が 0.134 と小さいこと．運賃は等級とほぼ連動しているので，等級の列が先に効果を持っていったのだと思う．係数は「ほかの特徴量を固定したときの効果」なので，相関の強い特徴量があると，片方の係数は小さく出やすい．

## 提出ファイルを作る

交差検証の平均がいちばん高かったモデルを，train 全体（891 行）で学習し直して test を予測する．交差検証の各回のモデルは train の 4/5 しか見ていないので，最後は全部のデータで学習したほうが有利になる．

In [ ]:
best_name = summary["平均"].drop("性別ルール").idxmax()
final_model = build_pipeline(models[best_name]).fit(X, y)
print("使うモデル:", best_name)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_model.predict(X_test),
})
path = save_submission(submission, "02_ml_baseline")
print("書き出し先:", path)
print(submission.head())

In [ ]:
# 予測した生存率が，train の生存率（0.384）から大きくずれていないか
print(f"test での予測生存率: {submission['Survived'].mean():.3f}")

# 性別ルールと予測が変わったのはどんな乗客か
by_sex = (test["Sex"] == "female").astype(int)
changed = test.assign(Predicted=submission["Survived"])[submission["Survived"] != by_sex]
print(f"性別ルールと予測が違う乗客: {len(changed)} 人")
print(changed.groupby(["Sex", "Pclass", "Predicted"]).size().rename("人数"))
print()
print(changed.loc[changed["Sex"] == "male", ["Name", "Age", "Pclass", "Predicted"]])

## 結果のまとめ

| | 交差検証 | LB |
|---|---|---|
| 性別ルール（01） | 0.787 | （未提出） |
| ランダムフォレスト（02） | 0.825 | （未提出） |

最後のセルを見ると，性別ルールから予測が変わったのは 36 人だった．3 等の女性 25 人が「死亡」に，男性 11 人が「生存」に変わっている．生存に変わった男性のほとんどは 13 歳以下の男の子で，名前の敬称は "Master"（男の子に使う敬称）．年齢を特徴量に入れたことで，「女性と子どもが先」の「子ども」の部分を拾えたことになる．予測した生存率が 0.330 と train（0.384）より低いのは，3 等の女性を死亡側に倒した影響が大きい．

### 次にやること

- 01 と 02 を提出して，交差検証と LB のスコアの差を見る．Titanic は交差検証より LB が低めに出るとよく言われるので，実際どのくらい差が出るのか確かめたい．
- 敬称（Mr / Mrs / Miss / Master）は，Age が欠けている人でも年齢層の手がかりになりそう．Discussion や他の人のノートブックで，どんな特徴量が使われているかを見てから取り入れる．